In [28]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from typing import TypedDict, Literal
from pydantic import BaseModel, Field

load_dotenv(override=True)

True

In [25]:
model = ChatOpenAI(
    base_url=os.getenv("OPENROUTER_SERVER"),
    model=os.getenv("MODEL_ID")
)
print(model.model_name)

response = model.invoke("Hello! Are you online?")
    
print("\n API is working! Here is the response:")
print("-" * 30)
print(response.content)

deepseek/deepseek-v4-flash

 API is working! Here is the response:
------------------------------
Hello! Yes, I'm online and ready to help you. How can I assist you today?


In [37]:
# define state schema and category-response scheme 

# state schema 
class CustomerTicket(TypedDict):
    ticket_text : str
    category : Literal["billing", "technical", "general"]
    response : str

# category-response    
class CategorySchema(BaseModel):
    category : Literal["billing", "technical", "general"] = Field(
        description="category of the support ticket" 
        )
    escalate : bool = Field(
        description="Is the request urgent; Yes: True, No: False"
    )

In [38]:
structured_model = model.with_structured_output(CategorySchema)

In [40]:
prompt = "I was charged twice for my subscription this month and I want a refund. This is very urgent."
print(structured_model.invoke(prompt).category, structured_model.invoke(prompt).escalate)

billing True


In [36]:
def classify_category(state : CustomerTicket)-> dict:
    
    prompt = f"Classify the following support ticket into one of: billing, technical, or general.\n\nTicket: {state['ticket_text']}"
    category = structured_model.invoke(prompt).category
    
    return {'category': category}